# UniZK: Figure 10 (hardware sensitivity on MVM), resumable

Reproduces **Figure 10** of *UniZK* (ASPLOS '25): MVM execution time, normalized to the default design (8 MB scratchpad, 32 VSAs, ~1 TB/s HBM), as each resource is scaled from below 1× up to 2×.

| Line | Points (normalized → setting) | Knob |
|---|---|---|
| SRAM Capacity | 1/8, 1/4, 1/2, 1, 2 → 1, 2, 4, 8, 16 MB | simulator `-r` |
| Array Count | 1/32 … 2 → 1, 2, 4, 8, 16, 32, 48, 64 VSAs | simulator `-t` |
| DRAM Bandwidth | 1/16 … 2 → 8, 16, 32, 64, 128, 256 HBM2 channels | RamSim config `channel:` |

That's 17 points from **12 simulator runs**: the bandwidth knob only affects RamSim, so one trace is streamed into 6 RamSim copies at once.
*Assumption:* the paper doesn't say how it scaled bandwidth; this scales the HBM2 channel count in RamSim (128 is the repo's default config).

**What makes this practical on Colab**
- **No trace files on disk:** traces are streamed through named pipes into RamSim as they're generated.
- **Checkpoints on Google Drive:** each finished point is saved immediately; re-running the notebook resumes and only redoes runs that were in flight.
- Runs in the background; the monitor cell can be stopped and restarted freely.


**Runtime:** Colab Pro **High-RAM CPU**. **RAM decides what's possible here.** Peak RAM is about 0.19 GB per simulated ms (measured on Fibonacci and a small MVM), so the paper's full MVM (320 ms) needs **~60 GB per run**. That's more than a High-RAM runtime's ~50 GB, so expect the full size to be OOM-killed on Colab. Each run takes about 3.5 h, so the 12 runs take ~42 h back to back.

Two ways to get a complete Figure 10:
1. **Colab, scaled MVM:** set `MVM_SIZE` smaller (start with 2000; if that is OOM-killed, try 1500). The runner reports it within minutes. Results go to a separate folder; the trends are comparable, but say in your report that it's a scaled workload.
2. **Full size on a ≥ 128 GB machine** (e.g. a university HPC node): the same runner via `run_unizk_hpc.sh`, 2 runs in parallel → about 1 day. Copy the state folder to Drive and plot with the Results cell here.

In [1]:
# ---- configuration ----
MVM_SIZE = 3000         # paper: 3000 x 3000 (~60 GB/run). On Colab try 2000. Separate results folder per size
STATE = f'/content/drive/MyDrive/unizk/fig10_mvm{MVM_SIZE}'
APP = 'mvm' if MVM_SIZE == 3000 else f'mvm_s{MVM_SIZE}'
MAX_PARALLEL = None     # None = auto; parallel runs only start once one MVM run's peak RAM is known and fits
RETRY_OOM = True
PLAN = {'plan': 'fig10', 'app': APP}

In [2]:
# Mount Drive (all results/checkpoints live there), check resources, get the code
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess, json, sys
os.makedirs(STATE, exist_ok=True)
print(subprocess.run("nproc; free -g | head -2; df -h /content | tail -1", shell=True, capture_output=True, text=True).stdout)
if not os.path.isdir('/content/UniZK'):
    subprocess.run('git clone -q https://github.com/tsinghua-ideal/UniZK.git /content/UniZK', shell=True, check=True)
print('repo ready; state dir:', STATE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
12
               total        used        free      shared  buff/cache   available
Mem:             167           2         159           0           6         164
overlay         236G   44G  193G  19% /

repo ready; state dir: /content/drive/MyDrive/unizk/fig10_mvm3000


The runner (written to `/content/unizk_runner.py`): patches, build, job scheduling, checkpoints, plotting. No need to edit it.

In [3]:
%%writefile /content/unizk_runner.py
#!/usr/bin/env python3
"""UniZK experiment runner for Figures 8 and 10 (resumable, Drive-checkpointed).

Key ideas
- Streaming traces: each simulator run writes its memory trace into a named pipe that
  RamSim reads concurrently, so no trace file ever touches disk and trace generation
  overlaps with DRAM simulation. One trace can be tee'd into several RamSim copies with
  different DRAM configs (used for the Figure 10 bandwidth sweep).
- Every finished data point is written to STATE/results/<key>.json right away.
  Re-running skips finished points, so a crash or runtime reset only loses the
  points that were in flight.
- Each job runs in its own folder, so several jobs can run in parallel if RAM allows.

CLI:
  python3 unizk_runner.py build  --state DIR --apps fibonacci,mvm
  python3 unizk_runner.py run    --state DIR --plan fig8|fig10 [--max-parallel N] [--mem-gb G]
  python3 unizk_runner.py status --state DIR --plan fig8|fig10
"""
import argparse, datetime, glob, json, os, re, shutil, signal, socket, subprocess, sys, threading, time

REPO = os.environ.get('UNIZK_REPO', '/content/UniZK')
WORK = os.environ.get('UNIZK_WORK', '/content/unizk_jobs')        # local scratch (fifos only)
TOOLCHAIN_MAIN = 'nightly-2026-10-02'   # tested: factorial, fibonacci, mvm build with it
TOOLCHAIN_V01 = 'nightly-2024-07-15'    # plonky2 v0.1 fork (sha256, ecdsa, img_crop) needs a 2024 nightly
V01_APPS = {'sha256', 'ecdsa', 'img_crop'}
SIM_NAME = {'img_crop': 'crop'}         # name the example passes to RamConfig::new
PATCH_VERSION = 'stream-v1'             # bump to invalidate cached binaries

# Paper reference values, digitized from the PDF (about +-1 percentage point / +-3 %).
PAPER_FIG8 = {  # (NTT, Poly, Hash) % of execution time
    'factorial': (15.4, 71.6, 13.0), 'fibonacci': (19.1, 51.4, 29.5), 'ecdsa': (13.4, 64.2, 22.4),
    'sha256': (13.8, 73.6, 12.6), 'img_crop': (15.0, 69.4, 15.6), 'mvm': (26.8, 50.0, 23.2)}
PAPER_FIG10 = {  # normalized parameter -> normalized execution time (MVM)
    'SRAM Capacity': {0.125: 2.40, 0.25: 1.62, 0.5: 1.25, 1: 1.0, 2: 0.956},
    'Array Count': {1/32: 6.14, 1/16: 3.48, 0.125: 2.16, 0.25: 1.50, 0.5: 1.17, 1: 1.0, 1.5: 0.943, 2: 0.913},
    'DRAM Bandwidth': {1/16: 10.98, 0.125: 6.12, 0.25: 3.07, 0.5: 1.84, 1: 1.0, 2: 0.70}}
FIG8_APPS = ['fibonacci', 'ecdsa', 'img_crop']   # small -> large
DEFAULT = dict(sram_mb=8, vsas=32, channels=128)
FIG10_SRAM = [1, 2, 4, 16]                  # MB   (default 8)
FIG10_VSAS = [1, 2, 4, 8, 16, 48, 64]       # VSAs (default 32)
FIG10_CHANNELS = [8, 16, 32, 64, 256]       # HBM2 channels (default 128 ~ 1 TB/s)


def log(msg):
    print(f'[{datetime.datetime.now():%H:%M:%S}] {msg}', flush=True)


def sh(cmd, check=True):
    r = subprocess.run(['bash', '-c', cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if check and r.returncode != 0:
        tail = '\n'.join(r.stdout.strip().splitlines()[-40:])
        raise RuntimeError(f'command failed ({r.returncode}): {cmd[:300]}\n--- last output ---\n{tail}')
    return r


def write_json(path, obj):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    tmp = f'{path}.tmp.{os.getpid()}.{threading.get_ident()}'
    with open(tmp, 'w') as f:
        json.dump(obj, f, indent=1)
    try:
        os.replace(tmp, path)
    except OSError:                       # some FUSE mounts dislike rename-over
        shutil.copy(tmp, path); os.remove(tmp)


def read_json(path, default=None):
    try:
        with open(path) as f:
            return json.load(f)
    except (OSError, ValueError):
        return default


# --------------------------------------------------------------------------- setup
def _patch(path, old, new, marker):
    s = open(path).read()
    if marker in s:
        return
    if old not in s:
        raise RuntimeError(f'patch target not found in {path}; repo layout changed?')
    open(path, 'w').write(s.replace(old, new, 1))


def apply_patches():
    """Idempotent source patches (see notebook text for what each one does)."""
    _patch(f'{REPO}/thirdparty/ramsim/src/base/utils.h',
           '#include <vector>', '#include <vector>\n#include <cstdint>', '<cstdint>')
    _patch(f'{REPO}/thirdparty/ramsim/src/frontend/impl/ramsim/trace_gen.cpp',
           '            file.clear();\n            file.seekg(0);\n',
           '            file.clear();\n'
           '            // streaming patch: a named pipe cannot seek (tellg() == -1); read it front to back\n'
           '            if (file.tellg() > 0) file.seekg(0);\n'
           '            file.clear();\n', 'streaming patch')
    _patch(f'{REPO}/src/config/ram_config.rs',
           '    pub fn run(&mut self) {\n        self.bin_file.flush().unwrap();\n',
           '    pub fn run(&mut self) {\n        self.bin_file.flush().unwrap();\n'
           '        // streaming patch: trace is a named pipe read by an external RamSim; close it (EOF) and return\n'
           '        if std::env::var_os("UNIZK_EXTERNAL_RAMSIM").is_some() {\n'
           '            let devnull = File::create("/dev/null").unwrap();\n'
           '            drop(std::mem::replace(&mut self.bin_file, BufWriter::new(devnull)));\n'
           '            return;\n        }\n', 'UNIZK_EXTERNAL_RAMSIM')
    _patch(f'{REPO}/src/util.rs',
           '                1 => {\n                    ENABLE_CONFIG.hash = true;\n                }',
           '                1 => {\n                    ENABLE_CONFIG.hash = true;\n                }\n'
           '                2 => {\n                    ENABLE_CONFIG.other = true;\n                }', '2 => {')


def make_scaled_mvm(size):
    """mvm_s<size>: the MVM example with a size x size matrix instead of 3000 x 3000."""
    name = f'mvm_s{size}'
    src = open(f'{REPO}/examples/mvm.rs').read()
    src = src.replace('const INPUT_SIZE: usize = 3000;', f'const INPUT_SIZE: usize = {size};')
    src = src.replace('const OUTPUT_SIZE: usize = 3000;', f'const OUTPUT_SIZE: usize = {size};')
    assert f'= {size};' in src
    open(f'{REPO}/examples/{name}.rs', 'w').write(src)
    return name


def base_app(app):
    return 'mvm' if app.startswith('mvm_s') else app


def bin_cache(state):
    return f'{state}/bin'


def _cargo_build(apps, v01):
    toml = 'Cargo.toml.v0.1' if v01 else 'Cargo.toml'
    tc = TOOLCHAIN_V01 if v01 else TOOLCHAIN_MAIN
    target = f'{REPO}/target_v01' if v01 else f'{REPO}/target_main'
    shutil.copy(f'{REPO}/configs/{toml}', f'{REPO}/Cargo.toml')
    t = open(f'{REPO}/Cargo.toml').read()
    if v01 and 'rust-version' not in t:   # let the resolver pick crate versions the 2024 toolchain can build
        open(f'{REPO}/Cargo.toml', 'w').write(t.replace('edition = "2021"', 'edition = "2021"\nrust-version = "1.80"', 1))
    exs = ' '.join(f'--example {a}' for a in apps)
    log(f'cargo build ({tc}): {", ".join(apps)}')
    sh(f'source $HOME/.cargo/env && cd {REPO} && rm -f Cargo.lock && '
       f'CARGO_RESOLVER_INCOMPATIBLE_RUST_VERSIONS=fallback cargo +{TOOLCHAIN_MAIN} generate-lockfile -q && '
       f'CARGO_TARGET_DIR={target} cargo +{tc} build -q -r {exs}')
    return {a: f'{target}/release/examples/{a}' for a in apps}


def build(state, apps):
    """Build (or restore from the Drive cache) the RamSim binary and the given examples."""
    cache = bin_cache(state)
    os.makedirs(cache, exist_ok=True)
    stamp = read_json(f'{cache}/stamp.json', {})
    ok = stamp.get('patch') == PATCH_VERSION
    ram_bin = f'{REPO}/thirdparty/ramsim/build/ramulator2'
    ram_lib = f'{REPO}/thirdparty/ramsim/libramulator.so'
    apply_patches()
    if ok and os.path.exists(f'{cache}/ramulator2') and os.path.exists(f'{cache}/libramulator.so'):
        os.makedirs(os.path.dirname(ram_bin), exist_ok=True)
        shutil.copy(f'{cache}/ramulator2', ram_bin); shutil.copy(f'{cache}/libramulator.so', ram_lib)
        os.chmod(ram_bin, 0o755)
        log('RamSim restored from cache')
    else:
        log('building RamSim (~3 min)')
        sh(f'cd {REPO}/thirdparty/ramsim && mkdir -p build && cd build && cmake .. > /dev/null && make -j$(nproc) > make.log 2>&1 '
           f'|| (tail -30 make.log; exit 1)')
        shutil.copy(ram_bin, f'{cache}/ramulator2'); shutil.copy(ram_lib, f'{cache}/libramulator.so')
        ok = False
    need = [a for a in apps if not (ok and os.path.exists(f'{cache}/{a}'))]
    for a in need:
        if a.startswith('mvm_s'):
            make_scaled_mvm(int(a[5:]))
    if need:
        if not os.path.exists(os.path.expanduser('~/.cargo/bin/rustup')):
            log('installing Rust (~1 min)')
            sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain none -q")
        for tc in {TOOLCHAIN_MAIN} | ({TOOLCHAIN_V01} if any(base_app(a) in V01_APPS for a in need) else set()):
            sh(f'source $HOME/.cargo/env && rustup -q toolchain install {tc} --profile minimal')
        for v01 in (False, True):
            group = [a for a in need if (base_app(a) in V01_APPS) == v01]
            if group:
                for a, path in _cargo_build(group, v01).items():
                    shutil.copy(path, f'{cache}/{a}')
    for a in apps:
        os.chmod(f'{cache}/{a}', 0o755)
    write_json(f'{cache}/stamp.json', {'patch': PATCH_VERSION, 'apps': sorted(set(apps) | set(stamp.get('apps', []))),
                                        'built': str(datetime.datetime.now())})
    log('build ready: ' + ', '.join(apps))


# --------------------------------------------------------------------------- plans
def plan_fig8(apps=None):
    jobs = []
    for app in apps or FIG8_APPS:
        for e, kind in [(0, 'NTT'), (1, 'Hash'), (2, 'Poly')]:
            key = f'{app}_e{e}'
            jobs.append(dict(key=key, app=app, args=['-e', str(e)],
                             variants=[dict(key=key, channels=DEFAULT['channels'])],
                             meta=dict(app=app, kernel=kind)))
    return jobs


def plan_fig10(app='mvm'):
    r0, t0 = DEFAULT['sram_mb'], DEFAULT['vsas']
    jobs = [dict(key='bw_group', app=app, args=['-r', str(r0), '-t', str(t0)],
                 variants=[dict(key=f'ch{c}', channels=c) for c in sorted(FIG10_CHANNELS + [DEFAULT['channels']])],
                 meta=dict(sweep='DRAM Bandwidth'))]
    jobs += [dict(key=f'sram{r}', app=app, args=['-r', str(r), '-t', str(t0)],
                  variants=[dict(key=f'sram{r}', channels=DEFAULT['channels'])], meta=dict(sweep='SRAM Capacity', value=r))
             for r in FIG10_SRAM]
    jobs += [dict(key=f'vsa{t}', app=app, args=['-r', str(r0), '-t', str(t)],
                  variants=[dict(key=f'vsa{t}', channels=DEFAULT['channels'])], meta=dict(sweep='Array Count', value=t))
             for t in FIG10_VSAS]
    return jobs


def load_plan(state):
    cfg = read_json(f'{state}/plan.json')
    if not cfg:
        raise SystemExit(f'no plan.json in {state}; run the notebook setup cell first')
    if cfg['plan'] == 'fig8':
        return cfg, plan_fig8(cfg.get('apps'))
    return cfg, plan_fig10(cfg.get('app', 'mvm'))


def job_done(state, job):
    return all(os.path.exists(f'{state}/results/{v["key"]}.json') for v in job['variants'])


# --------------------------------------------------------------------------- one job
def _parse_ramsim(path):
    txt = open(path, errors='replace').read() if os.path.exists(path) else ''
    g = lambda k: int(m.group(1)) if (m := re.search(rf'{k}:\s*(\d+)', txt)) else None
    return dict(cycles=g('memory_system_cycles'), reads=g('total_num_read_requests'),
                writes=g('total_num_write_requests'))


def _rss_mb(pid, field='VmHWM'):
    try:
        for line in open(f'/proc/{pid}/status'):
            if line.startswith(field):
                return int(line.split()[1]) // 1024
    except OSError:
        pass
    return None


def run_job(state, job, stop_event=None):
    app, key = job['app'], job['key']
    name = SIM_NAME.get(base_app(app), base_app(app))
    jd = f'{WORK}/{key}'
    shutil.rmtree(jd, ignore_errors=True)
    os.makedirs(f'{jd}/traces')
    if base_app(app) == 'img_crop':
        os.symlink(f'{REPO}/random_color.png', f'{jd}/random_color.png')
    main_fifo = f'{jd}/traces/{name}.bin'
    os.mkfifo(main_fifo)
    yaml_src = open(f'{REPO}/configs/{name}.yaml').read()
    multi = len(job['variants']) > 1
    procs, ram_logs, fifos = [], {}, []
    for v in job['variants']:
        fifo = f'{jd}/traces/{v["key"]}.bin' if multi else main_fifo
        if multi:
            os.mkfifo(fifo); fifos.append(fifo)
        y = re.sub(r'traceFileName: .*', f'traceFileName: {fifo}', yaml_src)
        y = re.sub(r'(\n\s*channel:)\s*\d+', rf'\g<1> {v["channels"]}', y)
        open(f'{jd}/{v["key"]}.yaml', 'w').write(y)
        ram_logs[v['key']] = f'{jd}/ramsim_{v["key"]}.log'
        procs.append(subprocess.Popen([f'{REPO}/thirdparty/ramsim/build/ramulator2', '-f', f'{jd}/{v["key"]}.yaml'],
                                      cwd=jd, stdout=open(ram_logs[v['key']], 'w'), stderr=subprocess.STDOUT))
    if multi:   # fan one trace out to every RamSim
        procs.append(subprocess.Popen(['bash', '-c', f'exec tee {" ".join(fifos)} < {main_fifo} > /dev/null'], cwd=jd))
    env = dict(os.environ, UNIZK_EXTERNAL_RAMSIM='1', RUST_LOG='info')
    sim_out = f'{jd}/sim.out'
    t0 = time.time()
    sim = subprocess.Popen([f'{bin_cache(state)}/{app}'] + job['args'], cwd=jd, env=env,
                           stdout=open(sim_out, 'w'), stderr=subprocess.STDOUT)
    status_path = f'{state}/status/{key}.json'
    peak, phase, sim_rc, sim_s = 0, 'generating trace + RamSim', None, None
    while True:
        if sim_rc is None:
            peak = max(peak, _rss_mb(sim.pid) or 0)
            wpid, wst, ru = os.wait4(sim.pid, os.WNOHANG)   # rusage gives the true peak RSS
            if wpid:
                sim_rc = sim.returncode = os.waitstatus_to_exitcode(wst)
                peak = max(peak, ru.ru_maxrss // 1024)
                sim_s = time.time() - t0
                phase = 'RamSim draining'
                if sim_rc != 0:
                    break
        if all(p.poll() is not None for p in procs) and sim_rc is not None:
            break
        if stop_event is not None and stop_event.is_set():
            sim_rc = 'stopped'; break
        write_json(status_path, dict(job=key, app=app, args=job['args'], phase=phase, host=socket.gethostname(),
                                     elapsed_min=round((time.time() - t0) / 60, 1), sim_peak_rss_gb=round(peak / 1024, 2),
                                     updated=str(datetime.datetime.now())))
        time.sleep(5)
    for p in procs + [sim]:
        if p.poll() is None:
            p.kill()
    wall = time.time() - t0
    # keep logs on Drive (small)
    ldir = f'{state}/logs/{key}'
    os.makedirs(ldir, exist_ok=True)
    for f in [sim_out] + list(ram_logs.values()) + glob.glob(f'{jd}/*.yaml'):
        try:
            shutil.copy(f, ldir)
        except OSError:
            pass
    try:
        os.remove(status_path)
    except OSError:
        pass
    parsed = {v['key']: _parse_ramsim(ram_logs[v['key']]) for v in job['variants']}
    shutil.rmtree(jd, ignore_errors=True)
    info = dict(job=key, app=app, args=job['args'], sim_rc=sim_rc, sim_peak_rss_gb=round(peak / 1024, 2),
                wall_min=round(wall / 60, 1), sim_min=round((sim_s or wall) / 60, 1),
                host=socket.gethostname(), finished=str(datetime.datetime.now()))
    if sim_rc != 0:
        info['reason'] = ('out of memory (killed)' if sim_rc in (-9, 137) else
                          'stopped' if sim_rc == 'stopped' else f'simulator exit {sim_rc}')
        return False, info
    results = {}
    for v in job['variants']:
        r = parsed[v['key']]
        if not r['cycles']:
            info['reason'] = f'no cycle count in RamSim log for {v["key"]}'
            return False, info
        results[v['key']] = dict(info, **r, variant=v['key'], channels=v['channels'], meta=job.get('meta', {}))
    for k, r in results.items():           # write only when every variant succeeded
        write_json(f'{state}/results/{k}.json', r)
    return True, info


# --------------------------------------------------------------------------- scheduler
def mem_total_gb():
    """Usable RAM: the smaller of physical RAM and this process's cgroup limit (SLURM/containers)."""
    total = next(int(l.split()[1]) / 2**20 for l in open('/proc/meminfo') if l.startswith('MemTotal'))
    for p in ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes'):
        try:
            v = open(p).read().strip()
            if v.isdigit():
                total = min(total, int(v) / 2**30)
        except OSError:
            pass
    return total


def estimate_gb(state, app):
    """Peak RAM of earlier runs of this app (results or OOM records); None if unknown."""
    vals = [r.get('sim_peak_rss_gb', 0) for r in map(read_json, glob.glob(f'{state}/results/*.json')) if r and r.get('app') == app]
    vals += [f.get('sim_peak_rss_gb', 0) for fs in map(read_json, glob.glob(f'{state}/failures/*.json')) if fs
             for f in fs if f.get('app') == app and 'memory' in f.get('reason', '')]
    return max(vals) * 1.15 if vals else None


def run_plan(state, max_parallel=None, mem_gb=None, retry_oom=False):
    cfg, jobs = load_plan(state)
    budget = mem_gb or mem_total_gb() * 0.9
    max_parallel = max_parallel or cfg.get('max_parallel') or max(1, (os.cpu_count() or 2) // 3)
    build(state, sorted({j['app'] for j in jobs}))
    for p in glob.glob(f'{state}/status/*.json'):   # left over from a crashed session
        os.remove(p)
    stop = threading.Event()
    signal.signal(signal.SIGTERM, lambda *a: stop.set())
    running = {}            # key -> (thread, est_gb, exclusive)
    results_lock = threading.Lock()
    finished_ok = {}

    def worker(job):
        try:
            ok, info = run_job(state, job, stop)
        except Exception as e:  # keep the scheduler alive
            ok, info = False, dict(job=job['key'], app=job['app'], reason=f'runner error: {e!r}')
        with results_lock:
            finished_ok[job['key']] = (ok, info)

    def attempts(job):
        return read_json(f'{state}/failures/{job["key"]}.json', [])

    log(f'plan {cfg["plan"]}: {len(jobs)} jobs, max_parallel={max_parallel}, RAM budget {budget:.0f} GB')
    while True:
        with results_lock:
            for k, (ok, info) in list(finished_ok.items()):
                th, est, excl = running.pop(k)
                if ok:
                    log(f'DONE  {k}  ({info["wall_min"]} min, peak RAM {info["sim_peak_rss_gb"]} GB)')
                elif stop.is_set():
                    log(f'STOPPED {k} (will rerun next time)')
                else:
                    info['ran_alone'] = excl
                    fails = read_json(f'{state}/failures/{k}.json', []) + [info]
                    write_json(f'{state}/failures/{k}.json', fails)
                    log(f'FAIL  {k}: {info.get("reason")}' + ('' if excl else ' (will retry alone)'))
                del finished_ok[k]
        if stop.is_set():
            log('stop requested'); break
        todo = []
        for j in jobs:
            if j['key'] in running or job_done(state, j):
                continue
            fails = attempts(j)
            if any(f.get('ran_alone') and 'memory' in f.get('reason', '') for f in fails) and not retry_oom:
                continue                       # does not fit in this runtime's RAM
            if len(fails) >= 3 and not retry_oom:
                continue
            todo.append(j)
        if not todo and not running:
            break
        oom_apps = {f.get('app') for fs in map(read_json, glob.glob(f'{state}/failures/*.json')) if fs
                    for f in fs if 'memory' in f.get('reason', '')}
        for j in todo:
            exclusive = j['app'] in oom_apps     # an OOM peak is only a lower bound: run this app alone
            est = estimate_gb(state, j['app'])
            if est is None or exclusive:
                fits = not running             # unknown size or previously OOM: run alone
                est = budget
            else:
                used = sum(e for _, e, _ in running.values())
                fits = (len(running) < max_parallel and used + est <= budget
                        and not any(x for _, _, x in running.values()))
            if fits:
                th = threading.Thread(target=worker, args=(j,), daemon=True)
                running[j['key']] = (th, est, est >= budget)
                log(f'START {j["key"]}  app={j["app"]} args={" ".join(j["args"])} '
                    f'variants={",".join(v["key"] for v in j["variants"])}'
                    + (f'  (est {est:.0f} GB)' if est < budget else '  (running alone)'))
                th.start()
                if est >= budget:
                    break
        time.sleep(10)
    log('all done' if not stop.is_set() else 'stopped')
    print_status(state)


# --------------------------------------------------------------------------- status / results
def collect(state):
    return {os.path.basename(p)[:-5]: r for p in glob.glob(f'{state}/results/*.json') if (r := read_json(p))}


def print_status(state):
    cfg, jobs = load_plan(state)
    res = collect(state)
    nvar = sum(len(j['variants']) for j in jobs)
    done = sum(1 for j in jobs for v in j['variants'] if v['key'] in res)
    print(f'\n=== {cfg["plan"]}: {done}/{nvar} data points done  ({sum(job_done(state, j) for j in jobs)}/{len(jobs)} jobs) ===')
    for p in sorted(glob.glob(f'{state}/status/*.json')):
        s = read_json(p)
        if s:
            print(f'  RUNNING {s["job"]:<14} {s["phase"]:<26} {s["elapsed_min"]:>6} min  peak RAM {s["sim_peak_rss_gb"]} GB'
                  f'  (updated {s["updated"][11:19]})')
    for j in jobs:
        if job_done(state, j):
            r = res[j['variants'][0]['key']]
            cyc = ', '.join(f'{v["key"]}={res[v["key"]]["cycles"]:,}' for v in j['variants'])
            print(f'  done    {j["key"]:<14} {r["wall_min"]:>6} min  peak RAM {r["sim_peak_rss_gb"]:>5} GB  cycles: {cyc}')
        else:
            fails = read_json(f'{state}/failures/{j["key"]}.json', [])
            if fails:
                print(f'  failed  {j["key"]:<14} x{len(fails)}: {fails[-1].get("reason")}'
                      f'{" (alone; skipped unless retry_oom)" if fails[-1].get("ran_alone") and "memory" in fails[-1].get("reason","") else ""}')


def fig8_table(state):
    res = collect(state)
    rows = {}
    for app in FIG8_APPS:
        parts = {k: res.get(f'{app}_e{e}', {}).get('cycles') for e, k in [(0, 'NTT'), (2, 'Poly'), (1, 'Hash')]}
        if all(parts.values()):
            tot = sum(parts.values())
            rows[app] = dict(**{f'{k}_cycles': v for k, v in parts.items()}, total_cycles=tot,
                             **{f'{k}_%': round(100 * v / tot, 1) for k, v in parts.items()},
                             **{f'paper_{k}_%': p for k, p in zip(['NTT', 'Poly', 'Hash'], PAPER_FIG8[app])})
    return rows


def fig10_table(state):
    res = collect(state)
    base = res.get('ch128', {}).get('cycles')
    if not base:
        return None, []
    rows = []
    for c in sorted(FIG10_CHANNELS + [128]):
        if f'ch{c}' in res:
            rows.append(('DRAM Bandwidth', c / 128, f'{c} ch', res[f'ch{c}']['cycles']))
    for r in sorted(FIG10_SRAM + [8]):
        k = 'ch128' if r == 8 else f'sram{r}'
        if k in res:
            rows.append(('SRAM Capacity', r / 8, f'{r} MB', res[k]['cycles']))
    for t in sorted(FIG10_VSAS + [32]):
        k = 'ch128' if t == 32 else f'vsa{t}'
        if k in res:
            rows.append(('Array Count', t / 32, f'{t} VSAs', res[k]['cycles']))
    return base, [dict(sweep=s, x=x, setting=lbl, cycles=c, time_ms=c / 1e6, normalized=round(c / base, 4),
                       paper=PAPER_FIG10[s].get(x)) for s, x, lbl, c in rows]


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('cmd', choices=['build', 'run', 'status'])
    ap.add_argument('--state', required=True)
    ap.add_argument('--apps', default='')
    ap.add_argument('--max-parallel', type=int)
    ap.add_argument('--mem-gb', type=float)
    ap.add_argument('--retry-oom', action='store_true')
    a = ap.parse_args()
    if a.cmd == 'build':
        build(a.state, [x for x in a.apps.split(',') if x])
    elif a.cmd == 'run':
        run_plan(a.state, a.max_parallel, a.mem_gb, a.retry_oom)
    else:
        print_status(a.state)


# --------------------------------------------------------------------------- plots
KCOL = {'NTT': '#fbe0c4', 'Poly': '#d5e8d4', 'Hash': '#dae8fc'}   # the paper's fills
APP_LABEL = {'factorial': 'Factorial', 'fibonacci': 'Fibonacci', 'ecdsa': 'ECDSA', 'sha256': 'SHA-256',
             'img_crop': 'Image Crop', 'mvm': 'MVM'}


def plot_fig8(state, out_dir=None, show=True):
    import matplotlib.pyplot as plt, numpy as np, csv
    out_dir = out_dir or f'{state}/out'
    os.makedirs(out_dir, exist_ok=True)
    rows = fig8_table(state)
    apps = [a for a in FIG8_APPS if a in rows] or []
    order = ['factorial', 'fibonacci', 'ecdsa', 'sha256', 'img_crop', 'mvm']   # the paper's x order
    apps = [a for a in order if a in rows]
    if not apps:
        print('no complete Figure 8 apps yet'); return rows
    with open(f'{out_dir}/fig8.csv', 'w', newline='') as f:
        w = csv.writer(f)
        w.writerow(['app'] + list(rows[apps[0]].keys()))
        for a in apps:
            w.writerow([a] + list(rows[a].values()))
    fig, ax = plt.subplots(figsize=(1.25 * len(apps) + 2.2, 3.6))
    x = np.arange(len(apps)); w = 0.36
    for off, src in [(-w / 2 - 0.01, 'ours'), (w / 2 + 0.01, 'paper')]:
        bottom = np.zeros(len(apps))
        for k in ['NTT', 'Poly', 'Hash']:
            vals = np.array([rows[a][f'{k}_%'] if src == 'ours' else rows[a][f'paper_{k}_%'] for a in apps])
            ax.bar(x + off, vals, w, bottom=bottom, color=KCOL[k], edgecolor='black', linewidth=0.8,
                   hatch='///' if src == 'paper' else None, label=k if src == 'ours' else None)
            if src == 'ours':
                for xi, v, b in zip(x + off, vals, bottom):
                    if v >= 8:
                        ax.text(xi, b + v / 2, f'{v:.0f}', ha='center', va='center', fontsize=7)
            bottom += vals
        for xi in x + off:
            ax.text(xi, 101.5, src, ha='center', va='bottom', fontsize=7, color='#555')
    ax.set_xticks(x, [APP_LABEL[a] for a in apps], rotation=25, ha='right')
    ax.set_ylabel('Execution Time Breakdown (%)'); ax.set_ylim(0, 108); ax.set_yticks([0, 50, 100])
    ax.yaxis.grid(True, ls='--', color='#ddd'); ax.set_axisbelow(True)
    ax.legend(title='Legend', loc='upper left', bbox_to_anchor=(1.0, 1.0), frameon=True)
    ax.set_title('Figure 8: breakdown by kernel type (ours vs paper, hatched)', fontsize=9)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{out_dir}/fig8.{ext}', dpi=200)
    if show:
        plt.show()
    plt.close(fig)
    return rows


F10_STYLE = {'SRAM Capacity': ('#3b82c4', 'o'), 'Array Count': ('#e0a526', 'D'), 'DRAM Bandwidth': ('#6b6b6b', 's')}


def plot_fig10(state, out_dir=None, show=True):
    import matplotlib.pyplot as plt, csv
    out_dir = out_dir or f'{state}/out'
    os.makedirs(out_dir, exist_ok=True)
    base, rows = fig10_table(state)
    if not rows:
        print('baseline (ch128) not finished yet'); return rows
    with open(f'{out_dir}/fig10.csv', 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

    def axes_fmt(ax):
        ax.set_yscale('log', base=2)
        ax.set_yticks([0.5, 1, 2, 4, 8]); ax.set_yticklabels(['0.5', '1', '2', '4', '8'])
        ax.set_ylim(0.5, 12); ax.set_xlim(0, 2.05); ax.set_xticks([0, 0.5, 1, 1.5, 2])
        ax.grid(True, ls='--', color='#ddd'); ax.set_axisbelow(True)
        ax.set_xlabel('Normalized Parameter')

    # (a) paper-style: our three lines
    fig, ax = plt.subplots(figsize=(4.6, 3.4))
    for s, (c, m) in F10_STYLE.items():
        pts = sorted((r['x'], r['normalized']) for r in rows if r['sweep'] == s)
        if pts:
            ax.plot(*zip(*pts), color=c, marker=m, mfc='white', ms=5, lw=1.5, label=s)
    axes_fmt(ax); ax.set_ylabel('Normalized Execution Time')
    ax.legend(title='Legend', fontsize=8, title_fontsize=8)
    ax.set_title('Figure 10 (ours): MVM sensitivity', fontsize=9)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{out_dir}/fig10.{ext}', dpi=200)
    if show:
        plt.show()
    plt.close(fig)

    # (b) one panel per sweep, ours vs paper
    fig, axs = plt.subplots(1, 3, figsize=(11, 3.3), sharey=True)
    for ax, (s, (c, m)) in zip(axs, F10_STYLE.items()):
        pts = sorted((r['x'], r['normalized']) for r in rows if r['sweep'] == s)
        pp = sorted(PAPER_FIG10[s].items())
        ax.plot(*zip(*pp), color='#999', marker=m, mfc='white', ms=4, lw=1.2, ls='--', label='paper (digitized)')
        if pts:
            ax.plot(*zip(*pts), color=c, marker=m, ms=5, lw=1.8, label='ours')
            for xv, yv in pts:
                ax.annotate(f'{yv:.2f}', (xv, yv), textcoords='offset points', xytext=(4, 4), fontsize=7)
        axes_fmt(ax); ax.set_title(s, fontsize=9); ax.legend(fontsize=7)
    axs[0].set_ylabel('Normalized Execution Time')
    fig.suptitle(f'Figure 10 vs paper (baseline: 8 MB, 32 VSAs, 128 HBM2 channels = {base:,} cycles)', fontsize=9)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(f'{out_dir}/fig10_vs_paper.{ext}', dpi=200)
    if show:
        plt.show()
    plt.close(fig)
    return rows

Overwriting /content/unizk_runner.py


In [4]:
# Patch + build (or restore cached binaries from Drive: ~1 min after the first time)
import os, sys, importlib, json
os.environ['UNIZK_REPO'] = '/content/UniZK'
sys.path.insert(0, '/content')
import unizk_runner as U; importlib.reload(U)
json.dump(PLAN, open(f'{STATE}/plan.json', 'w'), indent=1)
U.build(STATE, [APP])

[18:51:41] building RamSim (~3 min)
[18:53:24] cargo build (nightly-2026-10-02): mvm
[18:53:27] build ready: mvm


In [5]:
# Start the runner in the background (safe to re-run: it won't start a second copy).
# It skips every data point already saved in STATE/results, so after a crash just run
# the cells from the top again and it continues where it stopped.
import subprocess, os, time
PIDFILE = '/content/unizk_runner.pid'

def runner_alive():
    try:
        pid = int(open(PIDFILE).read())
        return 'unizk_runner.py' in open(f'/proc/{pid}/cmdline').read()
    except Exception:
        return False

if runner_alive():
    print('runner already running, pid', open(PIDFILE).read())
else:
    cmd = ['python3', '-u', '/content/unizk_runner.py', 'run', '--state', STATE]
    if MAX_PARALLEL: cmd += ['--max-parallel', str(MAX_PARALLEL)]
    if RETRY_OOM:    cmd += ['--retry-oom']
    p = subprocess.Popen(cmd, stdout=open(f'{STATE}/runner.log', 'a'), stderr=subprocess.STDOUT,
                         start_new_session=True, env=dict(os.environ, UNIZK_REPO='/content/UniZK'))
    open(PIDFILE, 'w').write(str(p.pid))
    time.sleep(5)
    print('started runner, pid', p.pid)
    print(''.join(open(f'{STATE}/runner.log').readlines()[-5:]))

started runner, pid 42701
[18:53:27] RamSim restored from cache
[18:53:27] build ready: mvm
[18:53:27] plan fig10: 12 jobs, max_parallel=4, RAM budget 150 GB
[18:53:27] START bw_group  app=mvm args=-r 8 -t 32 variants=ch8,ch16,ch32,ch64,ch128,ch256  (running alone)



In [ ]:
# Live progress. Interrupting this cell does NOT stop the runner; re-run it any time.
import time, subprocess
from IPython.display import clear_output
while True:
    clear_output(wait=True)
    print(time.strftime('%H:%M:%S'),
          subprocess.run(['python3', '/content/unizk_runner.py', 'status', '--state', STATE],
                         capture_output=True, text=True, env=dict(os.environ, UNIZK_REPO='/content/UniZK')).stdout)
    print('--- runner.log (last 12 lines) ---')
    print(''.join(open(f'{STATE}/runner.log').readlines()[-12:]))
    if not runner_alive():
        print('runner is not running (finished, stopped, or the runtime was reset).')
        break
    time.sleep(120)

18:55:32 
=== fig10: 0/17 data points done  (0/12 jobs) ===
  RUNNING bw_group       generating trace + RamSim     2.0 min  peak RAM 8.53 GB  (updated 18:55:27)

--- runner.log (last 12 lines) ---
[18:53:27] RamSim restored from cache
[18:53:27] build ready: mvm
[18:53:27] plan fig10: 12 jobs, max_parallel=4, RAM budget 150 GB
[18:53:27] START bw_group  app=mvm args=-r 8 -t 32 variants=ch8,ch16,ch32,ch64,ch128,ch256  (running alone)



## Results
Re-run any time; it plots whatever points are done (needs the baseline `ch128` first). Files go to `STATE/out/`: `fig10.png/pdf` (paper style), `fig10_vs_paper.png/pdf`, `fig10.csv`.

In [ ]:
import importlib; importlib.reload(U)
import pandas as pd
rows = U.plot_fig10(STATE)
pd.DataFrame(rows)

In [ ]:
# Optional tools
import glob, os, signal
def stop_runner():
    """Stop the background runner; in-flight points are discarded and rerun next time."""
    if runner_alive():
        os.killpg(int(open(PIDFILE).read()), signal.SIGTERM)
        print('stop signal sent')

def clear_failures(key=None):
    """Forget recorded failures (all, or one job key) so the runner retries them."""
    for p in glob.glob(f'{STATE}/failures/{key or "*"}.json'):
        os.remove(p); print('cleared', os.path.basename(p))

# stop_runner()
# clear_failures()